# Training Phi -- the configuration potential

Phi answers one question about a chess position: **is this a position of the shape where the
side to move goes wrong?**

It is trained on 130,874 Lichess puzzle positions in the 1500-2200 band -- each one a real
human of a known rating who actually failed to solve it -- against matched negatives sharing
material, phase, check status and mobility.

**No engine evaluation appears anywhere in the labels or the loss.**

### Before you run

* **Accelerator: a SINGLE GPU** (P100 for preference). Phi trains on `cuda:0` with no DDP, so on
  a T4x2 the second card idles for the whole run. The preflight below prints what you got.
* Add **both** datasets as inputs: the dataset archive and the code archive.
* Internet can stay **off**. Nothing here downloads anything.

### The gates

| gate | test | threshold |
|---|---|---|
| **F0** | material-only AUC on the loaded data | < 0.65 |
| **F1** | Phi held-out AUC | > 0.70 |
| **F2** | Phi AUC minus material AUC | >= 0.03 |

**F0 should print about 0.488.** That value was independently verified against this dataset
build. If it differs, you are not training on the data that was audited -- stop and check which
dataset version is mounted.

**A failed F2 is a data result, not a tuning problem.** Do not respond by changing
hyper-parameters.


## 1. Environment preflight

Fails loudly rather than training quietly on a CPU: a Kaggle session bills wall-clock for a
GPU notebook whether or not the card is used.


In [ ]:
import os, sys, subprocess, glob, json, zipfile, shutil
import torch

print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB  SM {p.major}.{p.minor}')
if torch.cuda.device_count() > 1:
    print('  NOTE: phi_net uses cuda:0 only. A second card will idle -- prefer a single GPU.')
if torch.cuda.is_available() and torch.cuda.get_device_properties(0).major < 8:
    print('  SM < 8.0: no hardware bfloat16. This code uses float16, which is correct here.')

print()
print('mounted inputs:')
for d in sorted(glob.glob('/kaggle/input/*')):
    print(' ', d, '->', sorted(os.listdir(d))[:8])


## 2. Locate the inputs and stage the code

Nothing is hard-coded to a dataset slug: the mounts are identified by what they contain.
Handles the archive being expanded by Kaggle or left intact, and refuses to guess if two
candidates match.


In [ ]:
INPUT_ROOT = os.environ.get('CSZERO_INPUT_ROOT', '/kaggle/input')

def _looks_like(d, filename):
    if os.path.exists(os.path.join(d, filename)):
        return True
    for z in glob.glob(os.path.join(d, '*.zip')):
        try:
            with zipfile.ZipFile(z) as zf:
                if any(os.path.basename(n) == filename for n in zf.namelist()):
                    return True
        except zipfile.BadZipFile:
            pass
    for sub in glob.glob(os.path.join(d, '*/')):
        if os.path.exists(os.path.join(sub, filename)):
            return True
    return False

def find_mount(filename, what):
    hits = [d for d in sorted(glob.glob(INPUT_ROOT + '/*')) if _looks_like(d, filename)]
    if not hits:
        raise SystemExit(f'ABORT: no mounted dataset contains {filename} ({what}). '
                         f'Add it as an input in the sidebar.')
    if len(hits) > 1:
        raise SystemExit(f'ABORT: {len(hits)} mounts contain {filename}: {hits}. '
                         f'Remove the one you do not want -- guessing between dataset '
                         f'versions is how a stale result gets reported as a real one.')
    print(f'{what}: {hits[0]}')
    return hits[0]

DATA_DIR = find_mount('train.npz', 'dataset')
CODE_DIR = find_mount('train.py', 'code')

# stage the code into a writable package directory
PKG = '/kaggle/working/phi_net'
shutil.rmtree(PKG, ignore_errors=True)
os.makedirs(PKG, exist_ok=True)
src = CODE_DIR
if not glob.glob(os.path.join(src, '*.py')):
    archives = glob.glob(os.path.join(src, '*.zip'))
    if archives:
        with zipfile.ZipFile(archives[0]) as zf:
            zf.extractall('/kaggle/working/_code')
        src = '/kaggle/working/_code'
    else:
        subs = [d for d in glob.glob(os.path.join(src, '*/')) if glob.glob(d + '*.py')]
        src = subs[0] if subs else src
for f in sorted(glob.glob(os.path.join(src, '*.py'))):
    shutil.copy(f, PKG)
staged = sorted(os.path.basename(f) for f in glob.glob(PKG + '/*.py'))
print('staged', len(staged), 'modules:', staged)
assert '__init__.py' in staged and 'train.py' in staged, 'code archive is incomplete'


## 3. Run the ladder

**B1** (100k rows, 15 epochs) asks only *does Phi learn anything at all*. It is diagnostic --
it stops the run only if F0 fails or Phi fails to beat piece-counting. **B2** is the full
training split, and F1 is judged after it, on the test split, in the next cell.

`-u` is not decoration: a notebook `!command` pipes stdout, so Python block-buffers it and an
epoch line would not appear for roughly ninety epochs. The run would look hung when it is fine.

The run deletes its own previous checkpoints first, so a crash leaves nothing that a later cell
could mistake for a result.


In [ ]:
os.chdir('/kaggle/working')
cmd = [sys.executable, '-u', '-m', 'phi_net.run_kaggle',
       '--data-dir', DATA_DIR,
       '--out-dir', '/kaggle/working/phi_runs']
print(' '.join(cmd), flush=True)
rc = subprocess.call(cmd, cwd='/kaggle/working')
print('\nexit code:', rc)
if rc != 0:
    raise SystemExit('the ladder did not finish -- read the output above before continuing')


## 4. Evaluate on the held-out test split

Run this **once**. The validation split is what the run tunes against; the test split is what
you report, and every look at it to make a decision makes it a little less held-out.

It refuses to score a checkpoint trained on a different dataset build.


In [ ]:
cmd = [sys.executable, '-u', '-m', 'phi_net.evaluate',
       '--checkpoint', '/kaggle/working/phi_runs/phi_b2.pt',
       '--data-dir', DATA_DIR]
rc = subprocess.call(cmd, cwd='/kaggle/working')
print('\nexit code:', rc)


## 5. Summary to bring home

Everything under `/kaggle/working/` is saved as this notebook's output version -- but only if
the session ends gracefully. For a long run use **Save & Run All (Commit)** rather than an
interactive tab, which dies on idle.


In [ ]:
for path in sorted(glob.glob('/kaggle/working/phi_runs/*.json')):
    with open(path) as f:
        d = json.load(f)
    print('=' * 60)
    print(os.path.basename(path))
    if 'best' in d:
        print('  best val AUC   ', round(d['best']['auc'], 4),
              'at epoch', d['best'].get('epoch'))
        print('  material AUC   ', round(d['material_auc'], 4))
        print('  gates passed   ', d['gates_passed'])
        print('  wall clock     ', round(d['total_seconds'], 1), 's',
              f"({round(d['total_seconds']/max(len(d['history']),1),2)} s/epoch)")
    if 'test_auc' in d:
        print('  TEST AUC       ', round(d['test_auc'], 4))
        print('  material AUC   ', round(d['material_auc'], 4))
        print('  per source     ', {k: round(v, 4) for k, v in d['per_source'].items()})
        print('  gates passed   ', d['gates_passed'])
print()
print('files to download:', sorted(os.path.basename(p) for p in
                                   glob.glob('/kaggle/working/phi_runs/*')))


## How to read the result

* **F1 passes** -- configurations are learnable from this representation. The steering pipeline
  has its potential function; next is stage B/C of `PLAN_CONFIGURATION_STEERING.md`.
* **F1 fails** -- configurations are not recoverable at 18 planes with this network. That is a
  real finding, not a failure of nerve. The response is a different representation (relational
  features, or BT3 activations), **not** tuning until the number moves.

Watch the per-source line as much as the headline. If Phi separates positives from the
*spent-tactic* negatives but not from real quiet play, it has learned "a tactic just finished
here" rather than "a tactic is available from here" -- the artefact the dataset was rebuilt to
remove, arriving through the model instead of the data.

Commit the metrics JSON to the repo (not the `.pt`), so the numbers are on the record.
